# ⚡ Notebook 4: The Cost of `fsync` and Group Commit

The WAL in notebook 1 calls `os.fsync` after **every** write. That's the safest thing to do — every `put()` is durable before it returns — but on real hardware, `fsync` is **expensive**. It forces the OS to push data all the way to the physical disk.

In this notebook we'll measure three strategies:

| Strategy | Durability | Speed |
| --- | --- | --- |
| 🟥 No fsync | Lose data on OS/power crash | Very fast |
| 🟨 fsync every write | Never lose acknowledged data | Slow |
| 🟩 Group commit (fsync every N, or every X ms) | Lose at most the last batch | Fast-ish |

This is the same trade-off every real database exposes: Postgres has `synchronous_commit`, MySQL/InnoDB has `innodb_flush_log_at_trx_commit`, Kafka has `acks` and `flush.ms`.

## Learning objectives
- Feel how much slower `fsync`-per-write is than a pure append.
- Implement **group commit**: batch N writes, then one `fsync`.
- Understand the durability cost of batching.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/write-ahead-log
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook).

In [ ]:
import os, json, tempfile, shutil, time

WORKDIR = tempfile.mkdtemp(prefix="wal_fsync_")
print("workdir:", WORKDIR)

N = 2000   # how many writes to do in each experiment

## 🏎️ Benchmark harness

Each experiment writes `N` tiny records to a fresh file. The only thing that changes is **when** we flush and fsync.

In [ ]:
def bench(label, path, write_fn):
    if os.path.exists(path): os.remove(path)
    t0 = time.perf_counter()
    write_fn(path)
    t1 = time.perf_counter()
    dur_ms = (t1 - t0) * 1000
    per_op_us = (dur_ms * 1000) / N
    ops_per_s = N / (t1 - t0)
    print(f"{label:<28} {dur_ms:8.1f} ms total | {per_op_us:8.1f} µs/op | {ops_per_s:10.0f} ops/s")

## 🟥 Strategy 1: no fsync, not even flush

Fastest, but a power loss can erase anything the OS hadn't written to disk yet (could be seconds of data).

In [ ]:
def write_no_fsync(path):
    with open(path, "a") as f:
        for i in range(N):
            f.write(json.dumps({"op": "put", "k": f"k{i}", "v": i}) + "\n")

bench("no fsync", os.path.join(WORKDIR, "a.log"), write_no_fsync)

## 🟨 Strategy 2: fsync every write (what notebook 1 did)

Strongest durability: every single write is on physical disk before it returns. Watch the ops/s drop.

In [ ]:
def write_fsync_every(path):
    with open(path, "a") as f:
        for i in range(N):
            f.write(json.dumps({"op": "put", "k": f"k{i}", "v": i}) + "\n")
            f.flush()
            os.fsync(f.fileno())

bench("fsync every write", os.path.join(WORKDIR, "b.log"), write_fsync_every)

## 🟩 Strategy 3: group commit (fsync every N writes)

We still append every write immediately, but only call `fsync` once per batch. If we crash, we lose at most the current batch — a tunable trade-off.

In a real system, batching is usually driven by **either** a batch size *or* a time budget (whichever comes first), so a lone writer doesn't wait forever.

In [ ]:
def write_group_commit(path, batch):
    with open(path, "a") as f:
        for i in range(N):
            f.write(json.dumps({"op": "put", "k": f"k{i}", "v": i}) + "\n")
            if (i + 1) % batch == 0:
                f.flush(); os.fsync(f.fileno())
        f.flush(); os.fsync(f.fileno())   # final flush for the tail

for batch in (10, 100, 1000):
    bench(f"group commit (every {batch})", os.path.join(WORKDIR, f"g{batch}.log"),
          lambda p, b=batch: write_group_commit(p, b))

## 🔍 What the numbers say

Numbers vary by hardware (SSD vs HDD, laptop vs server, and especially macOS vs Linux — macOS `fsync` is actually `F_FULLFSYNC`-light by default), but you should see a clear pattern:

- *fsync every write* is **much** slower than *no fsync*, often by 10–100×.
- *Group commit* recovers most of the speed while only risking the last un-synced batch.
- Doubling the batch size roughly halves the fsync overhead — but also doubles your worst-case data loss window.

## 🧭 How real databases expose this

- **PostgreSQL** `synchronous_commit = on|off|remote_write|remote_apply` controls whether a commit waits for the WAL to reach disk/replica.
- **MySQL / InnoDB** `innodb_flush_log_at_trx_commit = 1|0|2` — `1` = fsync per transaction (safest), `2` = fsync ~once per second.
- **SQLite** `PRAGMA synchronous = FULL|NORMAL|OFF`.
- **Apache Kafka** `flush.messages` / `flush.ms` tune how often the broker fsyncs the log.

They're all exposing the same WAL knob you just implemented by hand.

In [ ]:
shutil.rmtree(WORKDIR); print("cleaned up")

## ✅ Recap

- The WAL's durability story ultimately depends on `fsync` — without it, writes live in OS buffers that a power cut can erase.
- `fsync` is expensive, so naive `fsync`-per-write caps throughput at a few thousand ops/s on good SSDs, much less on HDDs.
- **Group commit** batches many writes into one `fsync`, trading a tiny, bounded durability window for a big throughput win. This is the default strategy in most production databases.